# 11. Multi-Agent Team Building

Now we **build** our first team.

We will create two agents with different jobs and put them in a
**`RoundRobinGroupChat`** so they take turns to finish a task together.

## Real-life analogy

Think of **writing a birthday card**:

- **Writer** writes a first draft.
- **Reviewer** checks it and says "APPROVE" when it is good.

They take turns until the card is approved. That is exactly what we will build.

## The recipe

| Step | What we do |
|------|-----------|
| 1 | Make the brain (model client) |
| 2 | Make **agent 1** (writer) |
| 3 | Make **agent 2** (reviewer) |
| 4 | Put them in a team that **takes turns** |
| 5 | Add a **stop rule** (stop when reviewer says APPROVE) |
| 6 | Run the team on a task |

In [1]:
from dotenv import load_dotenv
load_dotenv()

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

# Agent 1: the writer
writer = AssistantAgent(
    name="writer",
    model_client=model_client,
    system_message="You write a short birthday message. Improve it if the reviewer asks.",
)

# Agent 2: the reviewer
reviewer = AssistantAgent(
    name="reviewer",
    model_client=model_client,
    system_message="Review the message. If it is good, reply with the single word APPROVE. Otherwise suggest one change.",
)

# Stop when the reviewer says APPROVE
stop = TextMentionTermination("APPROVE")

# Build the team: they take turns (writer, reviewer, writer, ...)
team = RoundRobinGroupChat([writer, reviewer], termination_condition=stop)

result = await team.run(task="Write a warm birthday message for my friend Sam.")

# Print the whole conversation
for m in result.messages:
    print(f"[{m.source}] {m.content}\n")

[user] Write a warm birthday message for my friend Sam.

[writer] Happy Birthday, Sam! 🎉 On your special day, I just want to remind you how much you mean to me and everyone around you. Your kindness and laughter light up the world. I hope this year brings you all the joy and adventures you deserve. Cheers to another fabulous year ahead! Enjoy every moment!

[reviewer] APPROVE



## Key points to remember

- A **team** is built from two or more agents with different **roles**.
- **`RoundRobinGroupChat([a, b])`** makes them **take turns** in order.
- A **termination condition** tells the team when to **stop** (here: when "APPROVE" appears).
- Run the team with **`await team.run(task=...)`** — same `run` as a single agent.
- The result holds the **full conversation** in `result.messages`.